In [ ]:
import json 
import pandas as pd 
with open("owlapy_generated_qa.json") as f :
    queries = json.load(f)
queries


## subclassOf


### turtle syntax - full ontology

In [ ]:
import numpy as np
queries_df = pd.DataFrame(queries)
np.unique(queries_df['reasoning_type'],return_counts = True)

In [ ]:
print(f'SubClassOf Q&A  total: {len(queries)}')
for rt in ['subClassOf_direct', 'subClassOf_inferred', 'subClassOf_negative']:
    print(f'  {rt}: {sum(1 for q in queries if q["reasoning_type"]==rt)}')

In [ ]:
subclass_queries = queries_df[queries_df["reasoning_type"].isin([
    "subClassOf_direct",
    "subClassOf_inferred",
    "subClassOf_negative"
])]
subclass_queries

In [ ]:
print(f'SubClassOf Q&A  total: {len(queries)}')
for rt in ['subClassOf_direct', 'subClassOf_inferred', 'subClassOf_negative']:
    print(f'  {rt}: {sum(1 for q in queries if q["reasoning_type"]==rt)}')

In [ ]:
## campione per velocizzare l'esperimento

## --- campione BILANCIATO per classe: 50 domande totali ---
## (stessa ontologia ttl, stesso modello del run precedente; qui cambia solo il campione)

SEED = 42
N_TOT = 50

# 50 non e' divisibile per 3 -> 17 / 17 / 16
quotas = {
    "subClassOf_direct":   17,
    "subClassOf_inferred": 17,
    "subClassOf_negative": 16,
}
assert sum(quotas.values()) == N_TOT

parts = []
for rt, n in quotas.items():
    pool = subclass_queries[subclass_queries["reasoning_type"] == rt]
    assert len(pool) >= n, f"{rt}: disponibili {len(pool)}, richieste {n}"
    parts.append(pool.sample(n=n, random_state=SEED))

balanced_sample_df = (
    pd.concat(parts)
      .sample(frac=1, random_state=SEED)      # shuffle: i tipi non restano raggruppati
      .reset_index()
      .rename(columns={"index": "orig_index"})
      .drop(columns=["individual", "class"], errors="ignore")
)

# salvataggio in locale del campione effettivamente usato (riproducibilita')
balanced_sample_df.to_json(
    "subclass_balanced_sample_50.json", orient="records", indent=2, force_ascii=False
)
balanced_sample_df.to_csv("subclass_balanced_sample_50.csv", index=False)

print("totale:", len(balanced_sample_df))
print(balanced_sample_df["reasoning_type"].value_counts())
print(balanced_sample_df["answer"].value_counts())
balanced_sample_df.head()

In [ ]:
import re
## onto context da caricare : univ-bench-inst.ttl (in formato txt) / no inst

with open("univ-bench-inst.txt") as f: 
    lubm_onto = f.read()


# 2. Configurazione delle modalità di Ragionamento per Nemotron
# Mappiamo i trigger nativi di NVIDIA inserendo vincoli di profondità logica
thinking_modalities = {
    "off": "detailed thinking off",
    "on": "detailed thinking on"
}

# 3. Scelta della modalità (es. "on" per ragionamento profondo)
selected_modality = "on" 

# 4. Prompt di Sistema Strutturato, 
### modifcare questo per non considerare unresolvable in query subclassOf su Tbox
system_prompt = f"""You are an expert in semantic web with deep understanding of Ontology Web Language (OWL). {thinking_modalities[selected_modality]}

Your task is to analyze the provided OWL ontology and evaluate a natural language query against it.

CRITICAL OWL SEMANTICS RULES:
1. Remember that OWL operates under the Open World Assumption (OWA).
2. If the query logically follows from the ontology axioms the final answer is "True".
3. If the query directly contradicts the ontology axioms the final answer is "False".
4. If the ontology does not contain enough information to definitively prove or disprove the query, the final answer is "Unresolvable".

OUTPUT FORMAT:
You must strictly return your final response inside the following XML tags. Do not add any text or conversational filler outside these tags.

<reasoning_trace>
[Write here your step-by-step logical deduction, referencing the specific OWL classes, properties, or axioms used]
</reasoning_trace>
<final_answer>
[Strictly write only ONE of these three words: True, False, Unresolvable]
</final_answer>
"""
system_prompt_subclassOf = f"""You are an expert in semantic web with deep understanding of Ontology Web Language (OWL). {thinking_modalities[selected_modality]}

Your task is to analyze the provided OWL ontology and evaluate a natural language query against it.

CRITICAL OWL SEMANTICS RULES:
1. If the query logically follows from the ontology axioms the final answer is "True".
2. If the query directly contradicts the ontology axioms the final answer is "False".

OUTPUT FORMAT:
You must strictly return your final response inside the following XML tags. Do not add any text or conversational filler outside these tags.

<reasoning_trace>
[Write here your step-by-step logical deduction, referencing the specific OWL classes, properties, or axioms used]
</reasoning_trace>
<final_answer>
[Strictly write only ONE of these: True, False]
</final_answer>
"""


# 5. Definizione della query dell'utente
user_query = "Is a FullProfessor always a Faculty member?"

user_prompt = f"""ONTOLOGY CONTEXT:
{lubm_onto}

USER QUERY:
{user_query}

Analyze the context and query, then output the requested XML structure.
"""



In [ ]:
import re
import ollama
import ollama


user_query = subclass_queries.loc[0,"question"]
user_prompt = f"""ONTOLOGY CONTEXT:
{lubm_onto}

USER QUERY:
{user_query}

Analyze the context and query, then output the requested XML structure.
"""
print("user prompt: ", user_prompt)



def generate_ollama_ans(model, system_prompt, prompt, **kwargs):
    messages = []
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': prompt})
    
    # 2. Raccogli i parametri aggiuntivi (es. temperature, top_p) in 'options'
    # Usando **kwargs, qualsiasi parametro extra inserito nella funzione finirà qui automaticamente
    options = kwargs
    # print("using these options for generating the answer: ", options)
    
    response = ollama.chat(
        model=model,
        messages=messages,
        options=options
    )
    
    return response#response['message']['content']

_model = "qwen3.5:4b"



In [ ]:
## prima valutazione
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix
with open("results_qwen_3_5_4b_subclass_queries_owlapy.jsonl") as f: 
    qwen_results = [json.loads(i) for i in f]
print("sample length: ", len(qwen_results))
_sub_subclass_queries_df = subclass_queries.iloc[:len(qwen_results),:]
_gt = [str(i['answer_gt']).strip().lower() for i in qwen_results]
qwen_discrete_ans = [str(i['final_answer']).strip().lower() for i in qwen_results]
_qwen_sample_accuracy = accuracy_score(y_true=_gt,y_pred=qwen_discrete_ans)
print("accuracy qwen su sample di subclass reasoning: ", _qwen_sample_accuracy)

labels = sorted(set(_gt) | set(qwen_discrete_ans))

cm = confusion_matrix(
    y_true=_gt,
    y_pred=qwen_discrete_ans,
    labels=labels
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix - Qwen subclass reasoning - xml ")
plt.tight_layout()
plt.show()

#### test OWA e temperature - ttl

In [ ]:
subclass_queries.loc[125,"question"]

In [ ]:
## testare se false viene classificato solo con unresolvable (con xml succede questo)
reasoning_pattern = r"<reasoning_trace>(.*?)</reasoning_trace>"
answer_pattern = r"<final_answer>(.*?)</final_answer>"

user_query = subclass_queries.loc[125,"question"]
user_prompt = f"""ONTOLOGY CONTEXT:
{lubm_onto}

USER QUERY:
{user_query}

Analyze the context and query, then output the requested XML structure.
"""

temperature_ts = [0,0.2,0.5,0.7,0.9]

generated_ans = generate_ollama_ans(system_prompt = system_prompt_subclassOf, prompt= user_prompt, model=_model,temperature = 0.6)


In [ ]:
print("thinking: ", generated_ans['message'].thinking)
reasoning_match = re.search(reasoning_pattern, generated_ans['message']['content'], re.DOTALL)
answer_match = re.search(answer_pattern, generated_ans['message']['content'], re.DOTALL)
reasoning_trace = reasoning_match.group(1).strip() if reasoning_match else None
final_answer = answer_match.group(1).strip() if answer_match else None
print("reported thinking trace: \n", reasoning_trace)
print(f"Final Answer:\n{final_answer}")
print("############################## METRICS ############################### \n")
print("length of thinking: ", len(generated_ans['message'].thinking))
print("length of reported trace: ", len(reasoning_trace))
### testare per varie temperatures

In [ ]:
reasoning_pattern = r"<reasoning_trace>(.*?)</reasoning_trace>"
answer_pattern = r"<final_answer>(.*?)</final_answer>"

# 2. Eseguiamo la ricerca nella stringa
# re.DOTALL è fondamentale se il ragionamento contiene dei "vado a capo" (\n)
reasoning_match = re.search(reasoning_pattern, generated_ans, re.DOTALL)
answer_match = re.search(answer_pattern, generated_ans, re.DOTALL)

# 3. Estraiamo il testo pulito usando .strip() per rimuovere spazi o tweet di riga extra
reasoning_trace = reasoning_match.group(1).strip() if reasoning_match else None
final_answer = answer_match.group(1).strip() if answer_match else None

# --- Output dei risultati ---
print("=== DATI ESTRATTI ===")
print(f"Reasoning Trace:\n{reasoning_trace}\n")
print(f"Final Answer:\n{final_answer}")

In [ ]:
import os
import re
import json
import ollama

_model = "qwen3.5:4b"

reasoning_pattern = r"<reasoning_trace>(.*?)</reasoning_trace>"
answer_pattern = r"<final_answer>(.*?)</final_answer>"


def generate_ollama_ans(model, system_prompt, prompt, **kwargs):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    response = ollama.chat(
        model=model,
        messages=messages,
        options=kwargs,
    )

    return response["message"]["content"]


output_file = "results_qwen_3_5_4b_subclass_queries_owlapy.jsonl"

# ------------------------------------------------------------------
# Recupera le query già elaborate (resume)
# ------------------------------------------------------------------
processed_questions = set()

if os.path.exists(output_file):
    with open(output_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            try:
                record = json.loads(line)
                processed_questions.add(record["question"])
            except Exception:
                # Ignora eventuali righe corrotte
                continue

print(f"Trovate {len(processed_questions)} query già elaborate.")

# ------------------------------------------------------------------
# Apre il file in APPEND (non sovrascrive)
# ------------------------------------------------------------------
with open(output_file, "a", encoding="utf-8") as f:

    for i, row in subclass_queries.iterrows():

        user_query = row["question"]

        # Salta le query già presenti nel jsonl
        if user_query in processed_questions:
            print(f"[{i}] skipped")
            continue

        answer_gt = row["answer"]

        user_prompt = f"""ONTOLOGY CONTEXT:
{lubm_onto}

USER QUERY:
{user_query}

Analyze the context and query, then output the requested XML structure.
"""

        try:
            generated_ans = generate_ollama_ans(
                model=_model,
                system_prompt=system_prompt,
                prompt=user_prompt,
                temperature=0.6,
            )

            reasoning_match = re.search(
                reasoning_pattern,
                generated_ans,
                re.DOTALL,
            )

            answer_match = re.search(
                answer_pattern,
                generated_ans,
                re.DOTALL,
            )

            reasoning_trace = (
                reasoning_match.group(1).strip()
                if reasoning_match
                else None
            )

            final_answer = (
                answer_match.group(1).strip()
                if answer_match
                else None
            )

            record = {
                "question": user_query,
                "answer_gt": answer_gt,
                "reasoning_trace": reasoning_trace,
                "final_answer": final_answer,
            }

        except Exception as e:
            record = {
                "question": user_query,
                "answer_gt": answer_gt,
                "reasoning_trace": None,
                "final_answer": None,
                "error": str(e),
            }

        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()

        # Aggiorna il set anche durante l'esecuzione
        processed_questions.add(user_query)

        print(f"[{i}] processed")

In [ ]:
## prima valutazione

from sklearn.metrics import accuracy_score, confusion_matrix
with open("results_qwen_3_5_4b_subclass_queries_owlapy.jsonl") as f: 
    qwen_results = [json.loads(i) for i in f]
_sub_subclass_queries_df = subclass_queries.iloc[:len(qwen_results),:]
_gt = [str(i['answer_gt']).strip().lower() for i in qwen_results]
qwen_discrete_ans = [str(i['final_answer']).strip().lower() for i in qwen_results]
_qwen_sample_accuracy = accuracy_score(y_true=_gt,y_pred=qwen_discrete_ans)
print("accuracy qwen su sample di subclass reasoning: ", _qwen_sample_accuracy)

In [ ]:
wrong_instances = []

for idx, row in enumerate(qwen_results):
    gt = str(row["answer_gt"]).strip().lower()
    pred = str(row["final_answer"]).strip().lower()

    if gt != pred:
        wrong_instances.append({
            "index": idx,
            "question": row["question"],
            "ground_truth": row["answer_gt"],
            "prediction": row["final_answer"],
            "reasoning_trace": row["reasoning_trace"],
        })

print(f"{len(wrong_instances)} errori su {len(qwen_results)} esempi")

for err in wrong_instances[:10]:
    print("="*80)
    print("Index:", err["index"])
    print("Question:", err["question"])
    print("GT:", err["ground_truth"])
    print("Prediction:", err["prediction"])
    print("Reasoning:")
    print(err["reasoning_trace"])

In [ ]:
import numpy as np 
print(np.unique(_sub_subclass_queries_df['answer'],return_counts=True))
print(np.unique(_sub_subclass_queries_df['reasoning_type'],return_counts=True))
print("nell'intero dataset: ", np.unique(pd.DataFrame(subclass_queries)['answer'],return_counts=True))


In [ ]:
errors_df = _sub_subclass_queries_df.iloc[[i['index'] for i in wrong_instances],:]
errors_df['llm_ans'] = [i['prediction'] for i in wrong_instances]
errors_df  ## nana bisogna ripetere

#### esperimento su campione bilanciato (50 domande) con report esplicito degli assiomi — ttl

Stessa ontologia (`univ-bench-inst.txt`, turtle) e stesso modello (`qwen3.5:4b`) del run precedente.
Cambiano due cose:

1. il campione e' **bilanciato per classe** — 17 `subClassOf_direct`, 17 `subClassOf_inferred`, 16 `subClassOf_negative` (50 totali), salvato in `subclass_balanced_sample_50.json`/`.csv`;
2. il system prompt chiede al modello, oltre alla `reasoning_trace`, una **lista degli assiomi usati** in un blocco `<axioms>` (un assioma per riga, nella stessa notazione turtle degli assiomi groundtruth), cosi' da poterli estrarre e confrontare con l'output di `build_subclass_justification`.

In [ ]:
## nuovo system prompt: il modello deve riportare esplicitamente la LISTA degli assiomi usati
## (formato a una riga per assioma, cosi' da poterli estrarre e confrontare con gli assiomi groundtruth
##  prodotti da subclass_justification.build_subclass_justification)

import re
import os
import json
import ollama

## onto context: stessa ontologia ttl del run precedente
with open("univ-bench-inst.txt") as f:
    lubm_onto = f.read()

thinking_modalities = {"off": "detailed thinking off", "on": "detailed thinking on"}
selected_modality = "on"

system_prompt_subclassOf_report_axioms = f"""You are an expert in semantic web with deep understanding of Ontology Web Language (OWL). {thinking_modalities[selected_modality]}

Your task is to analyze the provided OWL ontology and evaluate a natural language subsumption query (TBox reasoning) against it.

CRITICAL OWL SEMANTICS RULES:
1. If the query logically follows from the ontology axioms the final answer is "True".
2. If the query does not follow from the ontology axioms the final answer is "False".
   Class subsumption over the TBox is decidable: the absence of a derivation is a sound
   non-entailment result, not an "unknown" caused by the Open World Assumption.

OUTPUT FORMAT:
You must strictly return your final response inside the following XML tags. Do not add any text or conversational filler outside these tags.

<reasoning_trace>
[Write here your step-by-step logical deduction, referencing the specific OWL classes, properties, or axioms used]
</reasoning_trace>
<axioms>
[Write here the list of the ontology axioms you used, one per line, following the rules below]
</axioms>
<final_answer>
[Strictly write only ONE of these: True, False]
</final_answer>

RULES FOR THE <axioms> SECTION:
- Write ONE axiom per line and nothing else on that line: no numbering, no bullet, no comment, no prose.
- Every line must use EXACTLY one of these two templates:
    :ClassA rdfs:subClassOf :ClassB
    :ClassA owl:equivalentClass [intersectionOf ( :ClassB ... )]
  Use the first one when the ontology asserts an explicit rdfs:subClassOf between two NAMED classes.
  Use the second one when :ClassA is defined by an owl:equivalentClass whose intersection contains the named class :ClassB.
- Write class names as the local name prefixed by ':' exactly as spelled in the ontology (e.g. :FullProfessor). Never write a full IRI and never write a blank node identifier.
- List ONLY the axioms that are strictly necessary to derive the answer (the minimal justification), in the order in which they are used in the derivation chain. Do not add axioms that are true but not needed.
- Do NOT write inference steps here (transitivity, "therefore X is a subclass of Y", ...): those belong to <reasoning_trace>. This section must contain only axioms asserted in the ontology.
- Never invent an axiom: every line must be literally verifiable in the ONTOLOGY CONTEXT.
- If no axiom supports the query, write exactly: NONE
"""

print(system_prompt_subclassOf_report_axioms)

In [ ]:
import time
from subclass_justification import (load_graph, build_named_subclass_edges,
                                    build_subclass_justification)

_model = "qwen3.5:4b"                       # stesso modello del run precedente
sample_file = "subclass_balanced_sample_50.json"
output_file = "results_qwen_3_5_4b_subclass_balanced50_axioms_ttl.jsonl"

reasoning_pattern = r"<reasoning_trace>(.*?)</reasoning_trace>"
axioms_pattern = r"<axioms>(.*?)</axioms>"
answer_pattern = r"<final_answer>(.*?)</final_answer>"


# il parser e' bracket-aware: unisce le righe quando il modello va a capo dentro
# una restrizione anonima spezzando un singolo assioma su piu' righe
from axiom_match import parse_axioms_block


def generate_ollama_ans(model, system_prompt, prompt, **kwargs):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    return ollama.chat(model=model, messages=messages, options=kwargs)


# rileggiamo il campione dal file salvato: cosi' il run usa esattamente
# il campione persistito in locale (riproducibile anche a kernel riavviato)
with open(sample_file) as f:
    balanced_sample = json.load(f)

g = load_graph("univ-bench-inst.ttl")
edges = build_named_subclass_edges(g)

processed_questions = set()
if os.path.exists(output_file):
    with open(output_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                processed_questions.add(json.loads(line)["question"])
            except Exception:
                continue
print(f"Trovate {len(processed_questions)} query gia' elaborate.", flush=True)

with open(output_file, "a", encoding="utf-8") as f:
    for i, row in enumerate(balanced_sample):
        user_query = row["question"]
        if user_query in processed_questions:
            print(f"[{i}] skipped", flush=True)
            continue

        answer_gt = row["answer"]
        gold = build_subclass_justification(g, edges, row["sub"], row["sup"])

        user_prompt = f"""ONTOLOGY CONTEXT:
{lubm_onto}

USER QUERY:
{user_query}

Analyze the context and query, then output the requested XML structure.
"""
        t0 = time.time()
        try:
            resp = generate_ollama_ans(
                model=_model,
                system_prompt=system_prompt_subclassOf_report_axioms,
                prompt=user_prompt,
                temperature=0.6,
            )
            content = resp["message"]["content"]
            thinking = getattr(resp["message"], "thinking", None)

            reasoning_match = re.search(reasoning_pattern, content, re.DOTALL)
            axioms_match = re.search(axioms_pattern, content, re.DOTALL)
            answer_match = re.search(answer_pattern, content, re.DOTALL)

            axioms_raw = axioms_match.group(1).strip() if axioms_match else None

            record = {
                "sample_id": i,
                "orig_index": row["orig_index"],
                "question": user_query,
                "sub": row["sub"],
                "sup": row["sup"],
                "reasoning_type": row["reasoning_type"],
                "answer_gt": answer_gt,
                "reasoning_trace": reasoning_match.group(1).strip() if reasoning_match else None,
                "axioms_raw": axioms_raw,
                "axioms": parse_axioms_block(axioms_raw),
                "final_answer": answer_match.group(1).strip() if answer_match else None,
                "gt_axioms": gold["axioms"],
                "gt_steps": gold["steps"],
                "thinking": thinking,
                "raw_response": content,
                "elapsed_s": round(time.time() - t0, 2),
            }
        except Exception as e:
            record = {
                "sample_id": i,
                "orig_index": row["orig_index"],
                "question": user_query,
                "sub": row["sub"],
                "sup": row["sup"],
                "reasoning_type": row["reasoning_type"],
                "answer_gt": answer_gt,
                "reasoning_trace": None,
                "axioms_raw": None,
                "axioms": None,
                "final_answer": None,
                "gt_axioms": gold["axioms"],
                "gt_steps": gold["steps"],
                "thinking": None,
                "raw_response": None,
                "elapsed_s": round(time.time() - t0, 2),
                "error": str(e),
            }

        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        processed_questions.add(user_query)
        print(f"[{i}] processed ({record['elapsed_s']}s) -> {record['final_answer']}", flush=True)

In [ ]:
## valutazione: risposta finale + confronto assiomi citati vs assiomi groundtruth
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix
from subclass_justification import load_graph, build_named_subclass_edges
from axiom_match import parse_axioms_block, canonical_set, canonical_axiom

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

output_file = "results_qwen_3_5_4b_subclass_balanced50_axioms_ttl.jsonl"
with open(output_file) as f:
    bal_results = [json.loads(line) for line in f if line.strip()]
print("risposte raccolte:", len(bal_results))

# ------------------------------------------------------------------
# NOTA SUL CONFRONTO
# Il template gold di subclass_justification usa un placeholder ('...') al posto delle
# restrizioni anonime:   :Employee owl:equivalentClass [intersectionOf ( :Person ... )]
# mentre il modello di norma le espande per intero. Le due scritture denotano lo STESSO
# assioma, quindi il confronto per stringa esatta misurerebbe la formattazione e non il
# contenuto. Si confrontano quindi le forme CANONICHE (axiom_match.canonical_axiom);
# la metrica per stringa esatta resta calcolata come riferimento.
# ------------------------------------------------------------------
g = load_graph("univ-bench-inst.ttl")
edges = build_named_subclass_edges(g)

ontology_canon = {
    canonical_axiom(e["axiom"])
    for node, lst in edges.items()
    if not node.endswith("__restrictions__")
    for e in lst
}
ontology_canon.discard(None)
print("assiomi (archi verso classi nominate) nell'ontologia:", len(ontology_canon))

norm_str = lambda a: " ".join(str(a).split()).lower()

# ------------------------------------------------------------------
# 1. accuratezza della final answer
# ------------------------------------------------------------------
_gt = [str(r["answer_gt"]).strip().lower() for r in bal_results]
_pred = [str(r["final_answer"]).strip().lower() for r in bal_results]
print("\naccuracy sul campione bilanciato:", accuracy_score(_gt, _pred))

labels = sorted(set(_gt) | set(_pred))
cm = confusion_matrix(_gt, _pred, labels=labels)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted label"); plt.ylabel("True label")
plt.title(f"{_model} - subclassOf campione bilanciato (ttl)")
plt.tight_layout(); plt.show()

# ------------------------------------------------------------------
# 2. metriche sugli assiomi citati
# ------------------------------------------------------------------
rows = []
for r in bal_results:
    cited_lines = parse_axioms_block(r["axioms_raw"])          # re-parse bracket-aware
    cited, unparsed, declarations = canonical_set(cited_lines)  # declaration = rumore
    gold, _, _ = canonical_set(r["gt_axioms"])

    tp = len(cited & gold)
    precision = tp / len(cited) if cited else (1.0 if not gold else 0.0)
    recall = tp / len(gold) if gold else None                   # indefinito sui negativi
    f1 = (2 * precision * recall / (precision + recall)
          if recall not in (None, 0) and precision else (None if recall is None else 0.0))

    hallucinated = [a for a in cited if a not in ontology_canon]

    # metrica per stringa esatta, solo come riferimento
    cited_str = {norm_str(a) for a in (cited_lines or [])}
    gold_str = {norm_str(a) for a in (r["gt_axioms"] or [])}
    tp_str = len(cited_str & gold_str)
    precision_str = tp_str / len(cited_str) if cited_str else (1.0 if not gold_str else 0.0)

    rows.append({
        "sample_id": r["sample_id"],
        "question": r["question"],
        "reasoning_type": r["reasoning_type"],
        "answer_gt": r["answer_gt"],
        "final_answer": r["final_answer"],
        "answer_ok": str(r["answer_gt"]).lower() == str(r["final_answer"]).lower(),
        "n_cited": len(cited),
        "n_gold": len(gold),
        "axiom_precision": precision,
        "axiom_recall": recall,
        "axiom_f1": f1,
        "exact_match": cited == gold,
        "n_hallucinated": len(hallucinated),
        "groundedness": (1 - len(hallucinated) / len(cited)) if cited else None,
        "n_declarations": len(declarations),   # ':X rdf:type owl:Class' -> rumore, non giustifica
        "n_unparsed": len(unparsed),
        "axiom_precision_stringa_esatta": precision_str,
        "hallucinated_axioms": "; ".join(str(a) for a in hallucinated),
    })

axiom_eval_df = pd.DataFrame(rows)

print("\n--- formato ---")
print("risposte con blocco <axioms> presente:",
      f"{sum(r['axioms_raw'] is not None for r in bal_results)}/{len(bal_results)}")
print("righe non interpretabili come assioma:", int(axiom_eval_df["n_unparsed"].sum()))

print("\n--- per reasoning_type (matching semantico) ---")
print(axiom_eval_df.groupby("reasoning_type").agg(
    n=("sample_id", "size"),
    answer_acc=("answer_ok", "mean"),
    axiom_precision=("axiom_precision", "mean"),
    axiom_recall=("axiom_recall", "mean"),
    exact_match=("exact_match", "mean"),
    groundedness=("groundedness", "mean"),
    mean_cited=("n_cited", "mean"),
    mean_gold=("n_gold", "mean"),
).round(3))

# sui subClassOf_negative il golden set e' VUOTO per costruzione (nessun assioma giustifica
# la subsumption): il recall e' indefinito e la precision misura la sovra-citazione.
pos = axiom_eval_df[axiom_eval_df["reasoning_type"] != "subClassOf_negative"]
print("\n--- solo casi positivi (direct + inferred) ---")
print("axiom precision (semantico):", round(pos["axiom_precision"].mean(), 3))
print("axiom recall    (semantico):", round(pos["axiom_recall"].mean(), 3))
print("axiom F1        (semantico):", round(pos["axiom_f1"].mean(), 3))
print("exact match     (semantico):", round(pos["exact_match"].mean(), 3))
print("[riferimento] precision per stringa esatta:",
      round(pos["axiom_precision_stringa_esatta"].mean(), 3))

neg = axiom_eval_df[axiom_eval_df["reasoning_type"] == "subClassOf_negative"]
print("\n--- solo casi negativi (golden set vuoto) ---")
print("casi in cui il modello non cita nulla:", int((neg["n_cited"] == 0).sum()), "/", len(neg))
print("media assiomi citati (sovra-citazione):", round(neg["n_cited"].mean(), 2))

print("\n--- assiomi allucinati (citati ma non presenti nell'ontologia) ---")
hal = axiom_eval_df[axiom_eval_df["n_hallucinated"] > 0]
print(hal[["sample_id", "question", "reasoning_type", "hallucinated_axioms"]].to_string(index=False)
      if len(hal) else "nessuno")

axiom_eval_df.to_csv("axiom_eval_qwen_3_5_4b_subclass_balanced50.csv", index=False)
axiom_eval_df.head(10)

### testing xml structure

In [ ]:
import re
## onto context da caricare : univ-bench-inst.ttl (in formato txt) / no inst

with open("univ-bench-inst_xml.txt") as f: 
    lubm_onto = f.read()


# 2. Configurazione delle modalità di Ragionamento per Nemotron
# Mappiamo i trigger nativi di NVIDIA inserendo vincoli di profondità logica
thinking_modalities = {
    "off": "detailed thinking off",
    "on": "detailed thinking on"
}

# 3. Scelta della modalità (es. "on" per ragionamento profondo)
selected_modality = "on" 

# 4. Prompt di Sistema Strutturato
system_prompt = f"""You are an expert in semantic web with deep understanding of Ontology Web Language (OWL). {thinking_modalities[selected_modality]}

Your task is to analyze the provided OWL ontology and evaluate a natural language query against it.

CRITICAL OWL SEMANTICS RULES:
1. Remember that OWL operates under the Open World Assumption (OWA).
2. If the query logically follows from the ontology axioms the final answer is "True".
3. If the query directly contradicts the ontology axioms the final answer is "False".
4. If the ontology does not contain enough information to definitively prove or disprove the query, the final answer is "Unresolvable".

OUTPUT FORMAT:
You must strictly return your final response inside the following XML tags. Do not add any text or conversational filler outside these tags.

<reasoning_trace>
[Write here your step-by-step logical deduction, referencing the specific OWL classes, properties, or axioms used]
</reasoning_trace>
<final_answer>
[Strictly write only ONE of these three words: True, False, Unresolvable]
</final_answer>
"""

# 5. Definizione della query dell'utente
user_query = "Is a FullProfessor always a Faculty member?"

user_prompt = f"""ONTOLOGY CONTEXT:
{lubm_onto}

USER QUERY:
{user_query}

Analyze the context and query, then output the requested XML structure.
"""

print(user_prompt)

In [ ]:
import os
import re
import json
import ollama

_model = "qwen3.5:4b"

reasoning_pattern = r"<reasoning_trace>(.*?)</reasoning_trace>"
answer_pattern = r"<final_answer>(.*?)</final_answer>"


def generate_ollama_ans(model, system_prompt, prompt, **kwargs):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    response = ollama.chat(
        model=model,
        messages=messages,
        options=kwargs,
    )

    return response["message"]["content"]


output_file = "results_qwen_3_5_4b_subclass_queries_owlapy_xml.jsonl"

# ------------------------------------------------------------------
# Recupera le query già elaborate (resume)
# ------------------------------------------------------------------
processed_questions = set()

if os.path.exists(output_file):
    with open(output_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            try:
                record = json.loads(line)
                processed_questions.add(record["question"])
            except Exception:
                # Ignora eventuali righe corrotte
                continue

print(f"Trovate {len(processed_questions)} query già elaborate.")

# ------------------------------------------------------------------
# Apre il file in APPEND (non sovrascrive)
# ------------------------------------------------------------------
with open(output_file, "a", encoding="utf-8") as f:

    for i, row in subclass_queries.iterrows():

        user_query = row["question"]

        # Salta le query già presenti nel jsonl
        if user_query in processed_questions:
            print(f"[{i}] skipped")
            continue

        answer_gt = row["answer"]

        user_prompt = f"""ONTOLOGY CONTEXT:
{lubm_onto}

USER QUERY:
{user_query}

Analyze the context and query, then output the requested XML structure.
"""

        try:
            generated_ans = generate_ollama_ans(
                model=_model,
                system_prompt=system_prompt,
                prompt=user_prompt,
                temperature=0.6,
            )

            reasoning_match = re.search(
                reasoning_pattern,
                generated_ans,
                re.DOTALL,
            )

            answer_match = re.search(
                answer_pattern,
                generated_ans,
                re.DOTALL,
            )

            reasoning_trace = (
                reasoning_match.group(1).strip()
                if reasoning_match
                else None
            )

            final_answer = (
                answer_match.group(1).strip()
                if answer_match
                else None
            )

            record = {
                "question": user_query,
                "answer_gt": answer_gt,
                "reasoning_trace": reasoning_trace,
                "final_answer": final_answer,
            }

        except Exception as e:
            record = {
                "question": user_query,
                "answer_gt": answer_gt,
                "reasoning_trace": None,
                "final_answer": None,
                "error": str(e),
            }

        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()

        # Aggiorna il set anche durante l'esecuzione
        processed_questions.add(user_query)

        print(f"[{i}] processed")

In [ ]:
print(generated_ans)

In [ ]:
_sub_subclass_queries_df

In [ ]:
## prima valutazione
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix
with open("results_qwen_3_5_4b_subclass_queries_owlapy_xml.jsonl") as f: 
    qwen_results_xml = [json.loads(i) for i in f]
print("sample length: ", len(qwen_results_xml))
# _sub_subclass_queries_df = subclass_queries.iloc[:len(qwen_results_xml),:]

_gt = [str(i['answer_gt']).strip().lower() for i in qwen_results_xml]
_gt_edit = [x if x != 'false' else 'unresolvable' for x in _gt]
qwen_discrete_ans = [str(i['final_answer']).strip().lower() for i in qwen_results_xml]
_qwen_sample_accuracy = accuracy_score(y_true=_gt_edit,y_pred=qwen_discrete_ans)
print("accuracy qwen su sample di subclass reasoning: ", _qwen_sample_accuracy)

labels = sorted(set(_gt_edit) | set(qwen_discrete_ans))

cm = confusion_matrix(
    y_true=_gt_edit,
    y_pred=qwen_discrete_ans,
    labels=labels
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix - Qwen subclass reasoning - xml ")
plt.tight_layout()
plt.show()

In [ ]:
[inst for inst in qwen_results_xml if inst['final_answer'] == 'False']

In [ ]:
print(qwen_results_xml[19]['reasoning_trace'])

##### evaluating natural language explanations

In [ ]:
### valtare differenza negli scores tra caso di explanations corretta e sbagliata (anche generata artificialmente)
from subclass_justification import load_graph, build_named_subclass_edges, build_subclass_justification
TTL = "univ-bench-inst.ttl"
g = load_graph(TTL)
edges = build_named_subclass_edges(g)
# To what extent do LLM-generated natural-language rationales faithfully represent valid OWL-based inference processes?
test_cases = [
("University", "Organization"), # atteso: True, asserted, 1 hop
("Institute", "Lecturer"),      # atteso: False (esempio della trace incollata)
("Lecturer", "Person"),         # atteso: True, catena asserted multi-hop
("Employee", "Person"),         # atteso: True, ma via equivalentClass, non asserted!
("Director", "Person"),         # atteso: False (caso limite: solo restrizione anonima)
("Chair", "Faculty"),           # atteso: True, asserted (Chair subClassOf Professor subClassOf Faculty)
("University", "University"),  # atteso: True, riflessivo
]

for sub, sup in test_cases:
    print("=" * 70)
    print(f"Is :{sub} a subClassOf :{sup} ?")
    res = build_subclass_justification(g, edges, sub, sup) ## chiama find_path che fa una BFS sul grafo, se non trova nulla la risposta è false
    # Se il path esiste, per ogni hop della catena controlla la fonte dell'arco (salvata da build_named_subclass_edges): se è "asserted" scrive uno 
    # step ":X ⊑ :Y — assioma"; se è "equivalentClass" scrive uno step che specifica che l'arco non è un rdfs:subClassOf esplicito ma deriva da una
    # definizione owl:equivalentClass/intersectionOf (più eventuali restrizioni associate). Ogni assioma citato viene anche accumulato nella lista axioms.
    print(f"-> answer: {res['answer']}")
    for s in res["steps"]:
        print(f"   {s}")
    if res["caveat"]:
        print(f"   [caveat] {res['caveat']}")
    print(f"   axioms (for scoring): {res['axioms']}")
    break


In [ ]:
res

In [ ]:
res['steps']

In [ ]:
import re


def normalize_axiom(axiom: str) -> str:
  """Rimuove namespace, prefissi, spazi e converte in minuscolo per un confronto flessibile."""
  # Rimuove prefissi come ':'
  cleaned = re.sub(r':', '', axiom)
  # Rimuove spazi vuoti e caratteri speciali
  cleaned = re.sub(r'\s+', '', cleaned).lower()
  return cleaned


def extract_axioms_from_text(text: str) -> list[str]:
  """Estrae pattern di assiomi OWL/RDFS dal testo generato dall'LLM."""
  patterns = [
      # Pattern Turtle: :ClassA rdfs:subClassOf :ClassB
      r'[:\w]+\s+rdfs:subClassOf\s+[:\w]+',
      # Pattern Simbolico: ClassA ⊑ ClassB
      r'[\w:]+\s*⊑\s*[\w:]+',
      # Pattern Equivale: :ClassA owl:equivalentClass ...
      r'[:\w]+\s+(?:owl:equivalentClass|≡)\s+\[?[^.\n\]]+\]?',
  ]

  extracted = []
  for pattern in patterns:
    matches = re.findall(pattern, text)
    extracted.extend(matches)

  return list(set(extracted))


def evaluate_reasoning(llm_response: str, golden_trace: list[str]):
  # 1. Normalizza la Golden Trace
  golden_normalized = [normalize_axiom(a) for a in golden_trace]

  # 2. Estrai e normalizza gli assiomi dalla risposta dell'LLM
  raw_llm_axioms = extract_axioms_from_text(llm_response)
  llm_normalized = [normalize_axiom(a) for a in raw_llm_axioms]

  # 3. Calcolo TP, FP, FN
  tp_set = set()
  fp_set = set()

  for llm_axiom in llm_normalized:
    # Verifichiamo la presenza esatta o parziale (es. per intersezioni complesse)
    if any(
        g in llm_axiom or llm_axiom in g for g in golden_normalized
    ):  # Match flessibile
      tp_set.add(llm_axiom)
    else:
      fp_set.add(llm_axiom)

  fn_set = set(golden_normalized) - set(
      g
      for g in golden_normalized
      if any(l in g or g in l for l in llm_normalized)
  )

  tp = len(tp_set)
  fp = len(fp_set)
  fn = len(fn_set)

  precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
  recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

  return {
      'precision': round(precision, 2),
      'recall': round(recall, 2),
      'true_positives': tp,
      'false_positives_unnecessary': fp_set,
      'false_negatives_missing': fn_set,
  }


# Esempio di utilizzo
llm_response = """
in order to identify if visiting professor is a subclass of professor, I need to proceed by steps
1. :VisitingProfessor rdfs:subClassOf :Professor
2. :Professor rdfs:subClassOf :Faculty
3. :Faculty rdfs:subClassOf :Employee
4. :Employee owl:equivalentClass [intersectionOf ( :Person ... )]
5. :VisitingProfessor ⊑ :Person  -- Assioma ridondante/estratto non presente in golden trace
"""

golden_trace = [
    ':VisitingProfessor rdfs:subClassOf :Professor',
    ':Professor rdfs:subClassOf :Faculty',
    ':Faculty rdfs:subClassOf :Employee',
    ':Employee owl:equivalentClass [intersectionOf ( :Person ... )]',
]

results = evaluate_reasoning(llm_response, golden_trace)
print(results)

In [ ]:
from subclass_justification import load_graph, build_named_subclass_edges, build_subclass_justification
TTL = "univ-bench-inst.ttl"
g = load_graph(TTL)
edges = build_named_subclass_edges(g)

## test explanation fornita da qwen_3_5_4b
test_resp_qwen = {"question": "Is VisitingProfessor a subclass of Person?", "answer_gt": "true", "reasoning_trace": "To determine if \"VisitingProfessor\" is a subclass of \"Person\", I need to trace the class hierarchy from VisitingProfessor upward:\n\n1. First, examine VisitingProfessor definition in the Classes section:\n   - :VisitingProfessor rdfs:subClassOf :Professor\n   \n2. Examine Professor's superclass:\n   - :Professor rdfs:subClassOf :Faculty\n   \n3. Examine Faculty's superclass:\n   - :Faculty rdfs:subClassOf :Employee\n\n4. Examine Employee's definition (this is crucial):\n   - :Employee has owl:equivalentClass with an intersection of two restrictions:\n     a) [rdf:type Person] AND b) [Person with worksFor some Organization restriction]\n   \n5. Since employee is equivalent to the intersection containing \"Person\", this means any instance that is an Employee MUST be a Person (in OWL, when you define a class as equivalent to an intersection including another class/property constraint, it inherits those type requirements).\n\n6. Following transitive subclass relationships:\n   - VisitingProfessor ⊑ Professor \n   - Professor ⊑ Faculty\n   - Faculty ⊑ Employee\n   - Since employee ≡ Person ∩ (worksFor restriction), then Employee ⊑ Person by definition of intersection equivalence\n   \n7. Therefore, the complete inheritance chain is:\n   - VisitingProfessor → Professor → Faculty → Employee → Person\n\nThis means all instances that are members of class \"VisitingProfessor\" must also be members of class \"Person\", making this statement logically entailed by the ontology axioms under OWL semantics (which operates on closed world for class inclusions).\n\nThe query is True because VisitingProfessor transitively inherits from Professor, which inherits from Faculty, which inherits from Employee, and since Employee's definition requires instances to be of type Person.", "final_answer": "True"}
print(test_resp_qwen['reasoning_trace'])
gold_trace = build_subclass_justification(g, edges, "VisitingProfessor", "Person")
print("gold trace obtained by reasoner: ", gold_trace)

print(
    "accuratezza final answer:",
    "corretto" if str(gold_trace['answer']).lower() == str(test_resp_qwen['answer_gt']).lower() else "non corretto"
)

print("steps del golden trace da seguire: ", gold_trace['axioms'])

## per confrontare meglio si potrebbe pensare ad un mapping dei simboli come ≡, ∩ e cosa significano in NL
print("reasoning score valutato tramite precision e recall con regex e confronto assiomi: ",evaluate_reasoning(llm_response=test_resp_qwen['reasoning_trace'], golden_trace=gold_trace['steps']))

# def evaluate_reasoning(explanation,golden_trace):
    # norm_explanation = "".join(explanation.strip().split()).lower()
    # normalized_axioms = ["".join(item.strip().split()).lower() for item in golden_trace]
    # for axiom in normalized_axioms:
    #     if axiom in norm_explanation:
    #         print("trovato: ", axiom)
    #     else:
    #         print("missing: ", axiom)  

In [ ]:
gold_trace

In [ ]:
print("risposta valutata: ", test_resp_qwen['reasoning_trace'])
_res_score = evaluate_reasoning(llm_response=test_resp_qwen['reasoning_trace'], golden_trace=gold_trace['steps'])
_res_score

In [ ]:
_model = "qwen3.5:4b"

reasoning_pattern = r"<reasoning_trace>(.*?)</reasoning_trace>"
answer_pattern = r"<final_answer>(.*?)</final_answer>"


def generate_ollama_ans(model, system_prompt, prompt, **kwargs):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    response = ollama.chat(
        model=model,
        messages=messages,
        options=kwargs,
    )

    return response

user_query = "Is :University a subClassOf :Organization ?"
user_prompt = f"""ONTOLOGY CONTEXT:
{lubm_onto}

USER QUERY:
{user_query}

Analyze the context and query, then output the requested XML structure.
"""
generated_ans = generate_ollama_ans(
    model=_model,
    system_prompt=system_prompt_subclassOf,
    prompt=user_prompt,
    temperature=0.6,
)

reasoning_match = re.search(
    reasoning_pattern,
    generated_ans['message']['content'],
    re.DOTALL,
)

answer_match = re.search(
    answer_pattern,
    generated_ans['message']['content'],
    re.DOTALL,
)

reasoning_trace = (
    reasoning_match.group(1).strip()
    if reasoning_match
    else None
)

final_answer = (
    answer_match.group(1).strip()
    if answer_match
    else None
)

record = {
    "question": user_query,
    "answer_gt": answer_gt,
    "reasoning_trace": reasoning_trace,
    "final_answer": final_answer,
}
thinking = generated_ans['message'].thinking
record

In [ ]:
_test_reasoning_trace = """
<reasoning_trace>
1.  **Identify the relevant class definitions**: I searched the provided ontology context for the definition of the class `:University`.
2.  **Locate subclass axioms**: In the "Classes" section, the following axiom was found for `:University`:
    ```xml
    ###  http://swat.cse.lehigh.edu/onto/univ-bench.owl#University
    :University rdf:type owl:Class ;
               rdfs:subClassOf :Organization ;
               rdfs:label "university" .
    ```
3.  **Verify the relationship**: The axiom explicitly states that `:University` is a subclass of `:Organization` (`rdfs:subClassOf :Organization`).
4.  **Check for contradictions**: I reviewed other class definitions and axioms to ensure there are no conflicting statements (e.g., disjoint classes or alternative parentage). No contradictory information was found regarding the superclass relationship between `:University` and `:Organization`.
5.  **Evaluate the query**: The user query asks "Is :University a subClassOf :Organization?". Since the ontology explicitly defines `:University` as a subclass of `:Organization`, the query logically follows from the axioms.
</reasoning_trace>
"""

In [ ]:
print(_test_reasoning_trace)

In [ ]:
reasoning_pattern = r"<reasoning_trace>(.*?)</reasoning_trace>"
answer_pattern = r"<final_answer>(.*?)</final_answer>"

reasoning_match = re.search(
    reasoning_pattern,
    _test_reasoning_trace,
    re.DOTALL,
)

answer_match = re.search(
    answer_pattern,
    _test_reasoning_trace,
    re.DOTALL,
)

In [ ]:
reasoning_match, reasoning_match.group()

#### testing explicitely reporting owl axioms / json structure with each step

In [ ]:
## altri criteri per valutare risposta: far citare anche il tipo di inferenza al llm
# Solidità logica (soundness): ogni passaggio deve seguire validamente dagli assiomi citati secondo la semantica OWL — non basta che i fatti citati siano veri, la catena deve essere effettivamente valida (è il difetto della trace B che abbiamo costruito: assioma corretto, inferenza finale non valida).

# Minimalità (justification, non "tutto quello che trovo"): la explanation ideale cita l'insieme minimo di assiomi necessari e sufficienti a derivare la conclusione — non uno di meno (sotto-citazione, un vero gap nella prova) e non uno di più (sovra-citazione, che è rumore e in un contesto di valutazione automatica peggiora anche la leggibilità e la score di precisione). Questo è esattamente il concetto di MinA nella letteratura DL, ed è quello che build_subclass_justification cerca di operazionalizzare.

# Groundedness: ogni assioma citato deve essere verificabile nel testo dell'ontologia data — citato letteralmente o parafrasato in modo fedele, mai "ricordato" dalla conoscenza pregressa del modello su cosa "di solito" è una University. Una explanation corretta nella conclusione ma che cita un assioma inventato (o non presente) è comunque una explanation cattiva.

# Consapevolezza del tipo di query: la explanation deve riconoscere esplicitamente se si tratta di un'inferenza TBox (subsumption, decidibile e completa) o ABox (instance-level, dove la OWA può davvero produrre indeterminatezza) — è il punto che abbiamo passato l'intera conversazione a correggere. Una buona explanation lo dice esplicitamente, non lo assume implicitamente.

# Trasparenza sul meccanismo di derivazione: non basta dire "X è sottoclasse di Y" — bisogna dire come: assioma diretto, transitività su una catena, o derivazione da owl:equivalentClass/restrizione (che semanticamente non è un rdfs:subClassOf esplicito, come abbiamo visto con Employee/Director). Questa distinzione non è pedanteria: cambia cosa il lettore (o il vostro scorer) deve verificare.

# Verifica di non-contraddizione sostanziale, non rituale: dichiarare "nessuna contraddizione trovata" ha senso solo se si è effettivamente controllato qualcosa di specifico (es. assiomi di disjointWith pertinenti alle classi coinvolte) — altrimenti è un'affermazione vuota, essa stessa non grounded.




### motivazioni per non introdurre json strutturato: 
# si rischia di misurare "sa compilare questo schema" più che "ragiona correttamente".
# modelli diversi hanno affidabilità molto diversa nel rispettare uno schema JSON rigido Inoltre perdi completamente il segnale di errore spontaneo: l'errore OWA della trace B (l'hedging ingiustificato) emerge naturalmente in prosa libera, ma potrebbe non emergere affatto in un campo owa_reasoning_note che il modello compila semplicemente ripetendo una frase-template plausibile, senza che rifletta il suo ragionamento reale su quella query specifica

In [ ]:
_model = "qwen3.5:4b"

## oppure: fornire risposta json con: final answer e qualcosa come: {
#   "steps": [
#     {
#       "type": "fact",
#       "content": "University ⊑ Organization",
#       "supported_by": "University rdfs:subClassOf Organization"
#     },
#     {
#       "type": "inference",
#       "rule": "subclass",
#       "conclusion": "University ⊑ Organization"
#     }
#   ]
# }



system_prompt_subclassOf_report_axioms ## TBD = f"""You are an expert in semantic web with deep understanding of Ontology Web Language (OWL). {thinking_modalities[selected_modality]}

Your task is to analyze the provided OWL ontology and evaluate a natural language query against it.

CRITICAL OWL SEMANTICS RULES:
1. If the query logically follows from the ontology axioms the final answer is "True".
2. If the query directly contradicts the ontology axioms the final answer is "False".

OUTPUT FORMAT:
You must strictly return your final response inside the following XML tags. Do not add any text or conversational filler outside these tags.

<reasoning_trace>
[Write here your step-by-step logical deduction, referencing the specific OWL classes, properties, or axioms used]
</reasoning_trace>
<axioms>

<final_answer>
[Strictly write only ONE of these: True, False]
</final_answer>
"""


def generate_ollama_ans(model, system_prompt, prompt, **kwargs):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    response = ollama.chat(
        model=model,
        messages=messages,
        options=kwargs,
    )

    return response

user_query = "Is :University a subClassOf :Organization ?"
user_prompt = f"""ONTOLOGY CONTEXT:
{lubm_onto}

USER QUERY:
{user_query}

Analyze the context and query, then output the requested XML structure.
"""
generated_ans = generate_ollama_ans(
    model=_model,
    system_prompt=system_prompt_subclassOf,
    prompt=user_prompt,
    temperature=0.6,
)

reasoning_match = re.search(
    reasoning_pattern,
    generated_ans['message']['content'],
    re.DOTALL,
)

answer_match = re.search(
    answer_pattern,
    generated_ans['message']['content'],
    re.DOTALL,
)

reasoning_trace = (
    reasoning_match.group(1).strip()
    if reasoning_match
    else None
)

final_answer = (
    answer_match.group(1).strip()
    if answer_match
    else None
)

record = {
    "question": user_query,
    "answer_gt": answer_gt,
    "reasoning_trace": reasoning_trace,
    "final_answer": final_answer,
}
thinking = generated_ans['message'].thinking
record

## testing domain range

In [ ]:
import json
import pandas as pd
import numpy as np

with open("owlapy_generated_qa_domain_range.json") as f:
    dr_queries = json.load(f)

dr_queries_df = pd.DataFrame(dr_queries)
print(np.unique(dr_queries_df["reasoning_type"], return_counts=True))
dr_queries_df

In [ ]:
import os
import re

import os
import re
import json
import ollama

_model = "qwen3.5:4b"

reasoning_pattern = r"<reasoning_trace>(.*?)</reasoning_trace>"
answer_pattern = r"<final_answer>(.*?)</final_answer>"


def generate_ollama_ans(model, system_prompt, prompt, **kwargs):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    response = ollama.chat(
        model=model,
        messages=messages,
        options=kwargs,
    )

    return response["message"]["content"]


output_file = "results_qwen_3_5_4b_domain_range_owlapy.jsonl"

# ------------------------------------------------------------------
# Recupera le query già elaborate (resume)
# ------------------------------------------------------------------
processed_questions = set()

if os.path.exists(output_file):
    with open(output_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            try:
                record = json.loads(line)
                processed_questions.add(record["question"])
            except Exception:
                # Ignora eventuali righe corrotte
                continue

print(f"Trovate {len(processed_questions)} query già elaborate.")

# ------------------------------------------------------------------
# Apre il file in APPEND (non sovrascrive)
# ------------------------------------------------------------------
with open(output_file, "a", encoding="utf-8") as f:

    for i, row in dr_queries_df.iterrows():

        user_query = row["question"]

        # Salta le query già presenti nel jsonl
        if user_query in processed_questions:
            print(f"[{i}] skipped")
            continue

        answer_gt = row["answer"]

        user_prompt = f"""ONTOLOGY CONTEXT:
{lubm_onto}

USER QUERY:
{user_query}

Analyze the context and query, then output the requested XML structure.
"""

        try:
            generated_ans = generate_ollama_ans(
                model=_model,
                system_prompt=system_prompt,
                prompt=user_prompt,
                temperature=0.6,
            )

            reasoning_match = re.search(
                reasoning_pattern,
                generated_ans,
                re.DOTALL,
            )

            answer_match = re.search(
                answer_pattern,
                generated_ans,
                re.DOTALL,
            )

            reasoning_trace = (
                reasoning_match.group(1).strip()
                if reasoning_match
                else None
            )

            final_answer = (
                answer_match.group(1).strip()
                if answer_match
                else None
            )

            record = {
                "question": user_query,
                "answer_gt": answer_gt,
                "reasoning_trace": reasoning_trace,
                "final_answer": final_answer,
            }

        except Exception as e:
            record = {
                "question": user_query,
                "answer_gt": answer_gt,
                "reasoning_trace": None,
                "final_answer": None,
                "error": str(e),
            }

        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()

        # Aggiorna il set anche durante l'esecuzione
        processed_questions.add(user_query)

        print(f"[{i}] processed")

In [ ]:
[str(i[1]['answer']).strip().lower() for i in _sub_dr_queries_df.iterrows()]

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

with open("results_qwen_3_5_4b_domain_range_owlapy.jsonl") as f:
    dr_results = [json.loads(i) for i in f]

_sub_dr_queries_df = dr_queries_df.iloc[:len(dr_results), :]
_gt_dr = [str(i[1]['answer']).strip().lower() for i in _sub_dr_queries_df.iterrows()]
dr_discrete_ans = [str(i['final_answer']).strip().lower() for i in dr_results]

_dr_accuracy = accuracy_score(y_true=_gt_dr, y_pred=dr_discrete_ans)
print("accuracy qwen su domain/range queries: ", _dr_accuracy)
print(confusion_matrix(_gt_dr, dr_discrete_ans))

In [ ]:
dr_results[0].keys()

In [ ]:
dr_wrong_instances = []

for idx, row in enumerate(dr_results):
    gt = str(row["answer_gt"]).strip().lower()
    pred = str(row["final_answer"]).strip().lower()

    if gt != pred:
        dr_wrong_instances.append({
            "index": idx,
            "question": row["question"],
            "ground_truth": row["answer_gt"],
            "prediction": row["final_answer"],
            "reasoning_trace": row["reasoning_trace"],
        })

print(f"{len(dr_wrong_instances)} errori su {len(dr_results)} esempi")

for err in dr_wrong_instances[:10]:
    print("=" * 80)
    print("Index:", err["index"])
    print("Question:", err["question"])
    print("GT:", err["ground_truth"])
    print("Prediction:", err["prediction"])
    print("Reasoning:")
    print(err["reasoning_trace"])

In [ ]:
# La mia idea iniziale è ottenere tramite la libreria python owlapy il tracing degli assiomi necessari 
# per la risoluzione di una query. Poi ottenere una risposta in linguaggio naturale  da un LLM.
#  Per confrontare la faithfulness e correttezza della risposta ho pensato a: usare una metrica judge, valutare se nella risposta siano stati riportati gli assiomi necessari (quindi poi calcolare precision e recall). 



## Equivalent Classes (owlapy)

List all equivalent class axioms using owlapy. Change `ONTO_PATH` to switch ontology.

In [ ]:
from owlapy.owl_ontology import SyncOntology
from owlapy.owl_reasoner import SyncReasoner

ONTO_PATH = "data/pizza.ttl"

onto = SyncOntology(ONTO_PATH)
reasoner = SyncReasoner(ontology=ONTO_PATH, reasoner="HermiT")
print(f"Ontology loaded: {onto.get_ontology_id()}")

In [ ]:
from owlapy.class_expression import OWLObjectIntersectionOf, OWLObjectSomeValuesFrom, OWLObjectAllValuesFrom, OWLObjectUnionOf, OWLObjectComplementOf

def format_ce(ce):
    if hasattr(ce, 'remainder') and ce.remainder:
        return ce.remainder
    if isinstance(ce, OWLObjectIntersectionOf):
        return ' ⊓ '.join(format_ce(op) for op in ce.operands())
    if isinstance(ce, OWLObjectUnionOf):
        return ' ⊔ '.join(format_ce(op) for op in ce.operands())
    if isinstance(ce, OWLObjectSomeValuesFrom):
        prop = ce.get_property()
        p = prop.remainder if hasattr(prop, 'remainder') else str(prop)
        return f'∃{p}.{format_ce(ce.get_filler())}'
    if isinstance(ce, OWLObjectAllValuesFrom):
        prop = ce.get_property()
        p = prop.remainder if hasattr(prop, 'remainder') else str(prop)
        return f'∀{p}.{format_ce(ce.get_filler())}'
    if isinstance(ce, OWLObjectComplementOf):
        return f'¬{format_ce(ce.get_operand())}'
    return str(ce)

for cls in sorted(onto.classes_in_signature(), key=lambda c: c.remainder):
    try:
        axioms = list(onto.equivalent_classes_axioms(cls))
    except AssertionError:
        continue
    for axiom in axioms:
        others = [ce for ce in axiom.class_expressions() if ce != cls]
        for ce in others:
            print(f"  {cls.remainder} ≡ {format_ce(ce)}")